In [2]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from TAM.tam import TAM
from TAM.qwen_utils import process_vision_info
from pathlib import Path

import os

/root/dl-hse-part2/hw3/TAM/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
prompt = 'Describe the picture. Your limit is 40-50 words.'
image_path = 'TAM/imgs/satellite_data'

def tam_for_qwen25(image_path, prompt, save_dir='vis_results'):
    os.makedirs(save_dir, exist_ok=True)
    
    model_name = "Qwen/Qwen2.5-VL-3B-Instruct"

    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto"
    )

    processor = AutoProcessor.from_pretrained(model_name)

    messages = [{"role": "user", "content": [{"type": "image", "image": image_path}, {"type": "text", "text": prompt}]}]
    
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt")
    inputs = inputs.to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        use_cache=True,
        output_hidden_states=True,
        return_dict_in_generate=True
    )
    
    generated_ids = outputs.sequences
    logits = [model.lm_head(feats[-1]) for feats in outputs.hidden_states]

    special_ids = {'img_id': [151652, 151653],
                   'prompt_id': [151653, [151645, 198, 151644, 77091]], 
                   'answer_id': [[198, 151644, 77091, 198], -1]}
    
    vision_shape = (inputs['image_grid_thw'][0, 1] // 2, inputs['image_grid_thw'][0, 2] // 2)
    vis_inputs = [[video_inputs[0][i] for i in range(0, len(video_inputs[0]))]] if isinstance(image_path, list) else image_inputs
    
    
    raw_map_records = []
    for i in range(len(logits)):
        img_map = TAM(
            generated_ids[0].cpu().tolist(),
            vision_shape,
            logits,
            special_ids,
            vis_inputs,
            processor,
            os.path.join(save_dir, str(i) + '.jpg'),
            i,
            raw_map_records,
            False)

In [8]:
img = Path(image_path) / "image2.png"

tam_for_qwen25(str(img), prompt, save_dir="imgs/vis_res")

/root/dl-hse-part2/hw3/TAM/.venv/lib/python3.12/site-packages/accelerate/utils/modeling.py:1566: UserWarning: Current model requires 4776 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 24.45it/s]
Some parameters are on the meta device because they were offloaded to the disk and cpu.


В качестве данных я взял разные фотографии местности, чтобы проверить, как метод TAM работает на таком не самом обычном домене.  
Поэкспериментировал я на 3 разных местностях: остров/океан, город/река, гористая местность.

<table>
  <tr>
    <td><img src="imgs/vis_res/45.jpg" width="400"/></td>
    <td><img src="imgs/vis_res/13.jpg" width="400"/></td>
  </tr>
  <tr>
    <td align="center"><b>Image 1: Good</b></td>
    <td align="center"><b>Image 2: Bad</b></td>
  </tr>
</table>

<table>
  <tr>
    <td><img src="imgs/vis_res/14.jpg" width="400"/></td>
    <td><img src="imgs/vis_res/8.jpg" width="400"/></td>
  </tr>
  <tr>
    <td align="center"><b>Image 1: Good</b></td>
    <td align="center"><b>Image 2: 50/50</b></td>
  </tr>
</table>

<table>
  <tr>
    <td><img src="imgs/vis_res/41.jpg" width="400"/></td>
    <td><img src="imgs/vis_res/17.jpg" width="400"/></td>
  </tr>
  <tr>
    <td align="center"><b>Image 1: Good</b></td>
    <td align="center"><b>Image 2: Bad</b></td>
  </tr>
</table>

Подводя итоги, с помощью метода TAM действительно можно удобно смотреть на то, как LLM связывает текстовое представление описания с изображением. Этот метод может помочь улучшить качество генерации ответов, а также избегать галлюцинаций (или их уменьшить)